# 🌲 5주차 실습 | AI 기초 강의
## 더 강한 모델 — Random Forest로 당뇨병 위험 예측하기

---

### 📌 오늘 실습 흐름

| STEP | 내용 | 4주차와 비교 |
|------|------|-------------|
| STEP 0 | 준비 — 라이브러리 + 한글 폰트 | 동일 |
| STEP 1 | 데이터 불러오기 | 새 데이터셋 (당뇨 환자) |
| STEP 2 | 전처리 (결측값 · 인코딩) | 4주차 복습 — 코드 거의 동일 |
| STEP 3 | 훈련 / 테스트 분리 | 동일 |
| STEP 4 | **DT vs RF 학습 & 성능 비교** | ⭐ 핵심 신규 |
| STEP 5 | **n_estimators 실험** | 4주차 depth 실험과 대응 |
| STEP 6 | **Feature Importance 비교** | DT vs RF 나란히 |

---

> **실행 방법**: 셀 왼쪽 ▶ 버튼 클릭 또는 `Shift + Enter`  
> ⚠️ **반드시 위에서 아래 순서대로 실행하세요!**

---
## 📦 STEP 0. 준비 — 가장 먼저 실행하세요

> ⚠️ 이 셀을 먼저 실행해야 아래 셀이 정상 작동합니다.  
> 한글 폰트 설치가 포함되어 있어 **30초 정도** 걸릴 수 있어요.

In [ ]:
# ── 라이브러리 불러오기 ───────────────────────────────────────────
import subprocess, glob, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Decision Tree (4주차에서 이미 배운 모델)
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Random Forest (오늘 배울 새 모델)
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
warnings.filterwarnings('ignore')

# ── 한글 폰트 설치 및 적용 ────────────────────────────────────────
subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-nanum'],
               capture_output=True)

nanum = [f for f in
         glob.glob('/usr/share/fonts/**/*.ttf', recursive=True)
         if 'NanumGothic.ttf' in f]

if nanum:
    fm.fontManager.addfont(nanum[0])
    prop = fm.FontProperties(fname=nanum[0])
    plt.rcParams['font.family'] = prop.get_name()
    print(f'✅ 한글 폰트 적용: {prop.get_name()}')
else:
    print('⚠️  나눔 폰트를 찾지 못했습니다.')

plt.rcParams['axes.unicode_minus'] = False
print('✅ 라이브러리 준비 완료! 이제 아래 셀을 순서대로 실행하세요.')

---
## 📂 STEP 1. 데이터 불러오기

오늘은 **당뇨병 환자 200명** 데이터를 사용합니다.

| 컬럼 | 타입 | 비고 |
|------|------|------|
| 나이 | **연속형** | 20~80세 |
| BMI | **연속형** | 체질량지수, 결측값 포함 ⚠️ |
| 혈당수치 | **연속형** | 공복혈당 (mg/dL) |
| 혈압 | **연속형** | 수축기 혈압 (mmHg) |
| 운동빈도 | **연속형** | 주당 횟수, 결측값 포함 ⚠️ |
| 가족력 | **범주형** | 있음/없음 → 인코딩 필요 |
| 당뇨위험 | **Label** | 0=저위험, 1=고위험 |

> 💡 4주차 student_data와 구조가 거의 동일합니다. 컬럼 이름만 달라요!

In [ ]:
# ── 데이터 불러오기 ───────────────────────────────────────────────
# GitHub raw URL에서 직접 불러옵니다
url = "https://raw.githubusercontent.com/runmc3812/AI_Data_-Analytics/refs/heads/main/5weeks/diabetes_data.csv"
df = pd.read_csv(url)

print(f'📋 데이터 크기: {df.shape[0]}행 × {df.shape[1]}열')
print()
print('── 컬럼 타입 ──────────────────────────────')
print(df.dtypes)
print()
print('── 처음 5행 ───────────────────────────────')
df.head()

In [ ]:
# ── 기초 통계 확인 ────────────────────────────────────────────────
# 연속형 Feature의 범위와 분포를 확인합니다
print('── 기초 통계 ──────────────────────────────')
df.describe().round(2)

---
## 🔍 STEP 2. 전처리

4주차와 동일한 순서로 진행합니다. 코드 구조가 익숙하게 느껴질 거예요!

**전처리 순서:**
1. 결측값 확인 → 평균으로 채우기
2. 범주형 인코딩 (가족력: 있음→1 / 없음→0)

> 💡 4주차와 달리 이번엔 이상치가 없어서 결측값 처리만 합니다.

In [ ]:
# ── 결측값 확인 ───────────────────────────────────────────────────
print('── 처리 전 결측값 ──────────────────────────')
print(df.isnull().sum())
print()
# BMI와 운동빈도에 결측값이 있어야 정상입니다
# 없다면 데이터 로딩을 다시 확인해주세요

In [ ]:
# ── 연속형 결측값 → 평균으로 채우기 ──────────────────────────────
# 4주차와 완전히 동일한 방법입니다!
for col in ['BMI', '운동빈도']:
    mean_val = df[col].mean()
    df[col] = df[col].fillna(mean_val)
    print(f'{col} 결측값 → 평균 {mean_val:.2f}로 채움')

print()
print('── 처리 후 결측값 ──────────────────────────')
print(df.isnull().sum())
print()
print('✅ 결측값 처리 완료!')

In [ ]:
# ── 범주형 인코딩 ─────────────────────────────────────────────────
# 가족력: 있음 → 1, 없음 → 0
# (4주차 전공계열 인코딩과 동일한 방식)

print('── 인코딩 전 ───────────────────────────────')
print(df['가족력'].value_counts())
print()

mapping = {'있음': 1, '없음': 0}
df['가족력'] = df['가족력'].map(mapping)

print('── 인코딩 후 ───────────────────────────────')
print(df['가족력'].value_counts().sort_index())
print()
print('✅ 인코딩 완료! 이제 모든 컬럼이 숫자입니다.')
df.head()

---
## ✂️ STEP 3. Feature / Label 분리 & 훈련/테스트 분리

4주차와 동일한 구조입니다. 컬럼 이름만 다릅니다!

- **X (Feature)**: 나이, BMI, 혈당수치, 혈압, 운동빈도, 가족력 (6개)
- **y (Label)**: 당뇨위험 (0=저위험 / 1=고위험)

In [ ]:
# ── Feature / Label 분리 ─────────────────────────────────────────
X = df[['나이', 'BMI', '혈당수치', '혈압', '운동빈도', '가족력']]
y = df['당뇨위험']

print(f'X (Feature) shape: {X.shape}  ← {X.shape[0]}명 × {X.shape[1]}개 Feature')
print(f'y (Label)   shape: {y.shape}  ← {y.shape[0]}명의 정답')
print()
print(f'고위험(1): {y.sum()}명  저위험(0): {(y==0).sum()}명')

# ── 훈련 / 테스트 분리 ───────────────────────────────────────────
# 4주차와 완전히 동일: 80% 훈련 / 20% 테스트
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print()
print(f'훈련 데이터: {len(X_train)}명 (80%)')
print(f'테스트 데이터: {len(X_test)}명 (20%)')
print()
print('✅ 분리 완료!')

---
## ⚔️ STEP 4. DT vs RF — 학습 & 성능 비교

오늘의 핵심 실습입니다!  
**같은 데이터, 두 모델** — 결과가 얼마나 다를까요?

| | Decision Tree | Random Forest |
|--|--|--|
| import | `sklearn.tree` | `sklearn.ensemble` |
| 클래스 | `DecisionTreeClassifier` | `RandomForestClassifier` |
| 핵심 파라미터 | `max_depth` | `n_estimators` |
| fit / predict | 동일 ✅ | 동일 ✅ |

In [ ]:
# ── Decision Tree 학습 ────────────────────────────────────────────
# 4주차에서 배운 그 모델 그대로입니다
dt_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

# ── Random Forest 학습 ────────────────────────────────────────────
# import 경로와 클래스 이름만 다를 뿐, 사용법은 완전히 동일!
rf_model = RandomForestClassifier(
    n_estimators=100,   # 나무를 100그루 만듭니다
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print('✅ 두 모델 학습 완료!')

In [ ]:
# ── 성능 지표 계산 및 비교 ────────────────────────────────────────
def get_scores(y_true, y_pred, name):
    """성능 지표를 딕셔너리로 반환하는 함수"""
    return {
        '모델': name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 3),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1 Score':  round(f1_score(y_true, y_pred, zero_division=0), 3),
    }

dt_scores = get_scores(y_test, dt_pred, 'Decision Tree')
rf_scores = get_scores(y_test, rf_pred, 'Random Forest')

# 비교 테이블 출력
result_df = pd.DataFrame([dt_scores, rf_scores]).set_index('모델')

print('=' * 52)
print(f'{"":20} {"Decision Tree":>14} {"Random Forest":>14}')
print('=' * 52)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1 Score']:
    dt_v = dt_scores[metric]
    rf_v = rf_scores[metric]
    # RF가 더 높으면 ↑ 표시
    arrow = '  ↑' if rf_v > dt_v else '   '
    print(f'  {metric:<18} {dt_v:>14.3f} {rf_v:>14.3f}{arrow}')
print('=' * 52)
print()
print('💡 ↑ 표시는 RF가 DT보다 높은 지표입니다.')

In [ ]:
# ── 성능 비교 시각화 ──────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
dt_vals = [dt_scores[m] for m in metrics]
rf_vals = [rf_scores[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

bars_dt = ax.bar(x - width/2, dt_vals, width,
                 label='Decision Tree', color='#FFB8A0', edgecolor='white')
bars_rf = ax.bar(x + width/2, rf_vals, width,
                 label='Random Forest', color='#4DAB87', edgecolor='white')

# 값 레이블 추가
for bar in bars_dt:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=11, color='#B85A2A')
for bar in bars_rf:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=11, color='#2D7A60')

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel('점수', fontsize=12)
ax.set_title('Decision Tree vs Random Forest — 성능 비교\n(같은 데이터, 모델만 바꿨습니다)',
             fontsize=13, fontweight='bold')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4, label='기준선 0.8')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 🎮 STEP 5. n_estimators 바꿔보기

4주차에서 `max_depth`를 바꿔봤던 것처럼,  
이번엔 `n_estimators`(나무 개수)를 바꿔가며 성능 변화를 확인합니다!

| n_estimators | 예상 결과 |
|--|--|
| `10` | 불안정 — 나무가 너무 적음 |
| `50` | 안정화 시작 |
| `100` | 적절한 균형 ✅ |
| `200` | 거의 동일하지만 더 느림 |

> 💡 `max_depth`와 달리 `n_estimators`를 늘려도 **과적합이 생기지 않습니다!**  
> 이게 Random Forest의 큰 장점이에요.

In [ ]:
# ✏️ 이 숫자를 바꿔보세요!
n_trees = 100

# ── 아래는 수정하지 마세요 ─────────────────────────────────────
m = RandomForestClassifier(n_estimators=n_trees, random_state=42)
m.fit(X_train, y_train)

train_acc = m.score(X_train, y_train)
test_acc  = m.score(X_test, y_test)
gap       = train_acc - test_acc

print(f'n_estimators = {n_trees}')
print(f'  훈련 정확도: {train_acc:.3f}  ({train_acc*100:.1f}%)')
print(f'  테스트 정확도: {test_acc:.3f}  ({test_acc*100:.1f}%)')
print(f'  차이 (훈련-테스트): {gap:.3f}')

if gap > 0.1:
    print('  ⚠️  훈련·테스트 차이가 큽니다.')
elif gap < 0.05:
    print('  ✅  훈련·테스트가 균형 잡혀 있습니다.')
else:
    print('  🟡  약간의 차이가 있습니다.')

In [ ]:
# ── n_estimators 범위별 성능 곡선 ────────────────────────────────
# 4주차 depth 실험과 동일한 구조로 비교해보세요!
estimator_range = [5, 10, 20, 50, 100, 150, 200]
train_accs = []
test_accs  = []

for n in estimator_range:
    m = RandomForestClassifier(n_estimators=n, random_state=42)
    m.fit(X_train, y_train)
    train_accs.append(m.score(X_train, y_train))
    test_accs.append(m.score(X_test, y_test))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 왼쪽: RF n_estimators 곡선 ───────────────────────────────────
axes[0].plot(estimator_range, train_accs, 'o-', color='#4DAB87',
             linewidth=2.5, markersize=7, label='훈련 정확도')
axes[0].plot(estimator_range, test_accs, 's-', color='#5BA8E0',
             linewidth=2.5, markersize=7, label='테스트 정확도')
if n_trees in estimator_range:
    axes[0].axvline(x=n_trees, color='gray', linestyle='--',
                    alpha=0.6, label=f'현재 n={n_trees}')
axes[0].set_xlabel('n_estimators (나무 개수)', fontsize=12)
axes[0].set_ylabel('정확도', fontsize=12)
axes[0].set_title('RF: n_estimators에 따른 정확도\n→ 늘려도 과적합이 없음!',
                  fontsize=12, fontweight='bold')
axes[0].set_ylim(0.7, 1.02)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# ── 오른쪽: DT depth 곡선 (4주차 결과와 비교) ────────────────────
depths = list(range(1, 16))
dt_train_accs = []
dt_test_accs  = []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    dt_train_accs.append(m.score(X_train, y_train))
    dt_test_accs.append(m.score(X_test, y_test))

axes[1].plot(depths, dt_train_accs, 'o-', color='#E8845A',
             linewidth=2.5, markersize=7, label='훈련 정확도')
axes[1].plot(depths, dt_test_accs, 's-', color='#E05C5C',
             linewidth=2.5, markersize=7, label='테스트 정확도')
best_dt_depth = depths[dt_test_accs.index(max(dt_test_accs))]
axes[1].axvspan(best_dt_depth + 0.5, 15.5,
                alpha=0.08, color='red')
axes[1].annotate('과적합\n구간', xy=(13, 0.75),
                 fontsize=10, color='#E05C5C', fontweight='bold')
axes[1].set_xlabel('max_depth', fontsize=12)
axes[1].set_ylabel('정확도', fontsize=12)
axes[1].set_title('DT: depth에 따른 정확도 (4주차 복습)\n→ 깊어질수록 과적합 발생!',
                  fontsize=12, fontweight='bold')
axes[1].set_ylim(0.5, 1.05)
axes[1].set_xticks(depths)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('RF(왼쪽) vs DT(오른쪽) — 파라미터 실험 비교',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n📌 RF: n_estimators가 커져도 테스트 정확도가 안정적으로 유지됩니다.')
print(f'📌 DT: depth가 깊어지면 테스트 정확도가 뚝 떨어집니다.')
print(f'    → 이것이 Random Forest의 핵심 강점!')

---
## 📊 STEP 6. Feature Importance 비교

DT와 RF가 **같은 데이터**에서 어떤 Feature를 더 중요하게 봤을까요?

- **DT**: 단 1개 트리 기준 → 데이터 변화에 민감할 수 있음
- **RF**: 수백 개 트리의 평균 → 더 안정적이고 신뢰할 수 있음

> 💡 보건의료 관점에서 어떤 Feature가 당뇨 위험에 가장 영향을 주는지 확인해보세요!

In [ ]:
# ── Feature Importance 텍스트 비교 ───────────────────────────────
feat_names = list(X.columns)
dt_imp = dt_model.feature_importances_
rf_imp = rf_model.feature_importances_

# RF 중요도 기준으로 정렬
sorted_idx = np.argsort(rf_imp)[::-1]

print('── Feature Importance 비교 ─────────────────────────')
print(f'{"Feature":<10}  {"Decision Tree":>14}  {"Random Forest":>14}')
print('-' * 44)
for i in sorted_idx:
    dt_bar = '█' * int(dt_imp[i] * 30)
    rf_bar = '█' * int(rf_imp[i] * 30)
    print(f'{feat_names[i]:<10}  {dt_imp[i]:>6.3f} {dt_bar:<10}  '
          f'{rf_imp[i]:>6.3f} {rf_bar}')
print('-' * 44)
print(f'{"합계":<10}  {dt_imp.sum():>14.3f}  {rf_imp.sum():>14.3f}')
print()
print(f'🔍 DT 1위: {feat_names[np.argmax(dt_imp)]}')
print(f'🔍 RF 1위: {feat_names[np.argmax(rf_imp)]}')

In [ ]:
# ── Feature Importance 시각화 ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_dt = ['#FFB8A0'] * len(feat_names)
colors_rf = ['#4DAB87'] * len(feat_names)

# 1위 Feature는 진하게 강조
colors_dt[np.argmax(dt_imp)] = '#B85A2A'
colors_rf[np.argmax(rf_imp)] = '#2D7A60'

# DT Feature Importance
dt_sorted = np.argsort(dt_imp)
axes[0].barh([feat_names[i] for i in dt_sorted],
             [dt_imp[i] for i in dt_sorted],
             color=[colors_dt[i] for i in dt_sorted],
             edgecolor='white')
axes[0].set_title('Decision Tree\nFeature Importance',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('중요도', fontsize=11)
axes[0].set_xlim(0, max(dt_imp) * 1.3)
axes[0].grid(True, alpha=0.3, axis='x')
for i, (feat, val) in enumerate(zip([feat_names[j] for j in dt_sorted],
                                     [dt_imp[j] for j in dt_sorted])):
    axes[0].text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=11)

# RF Feature Importance
rf_sorted = np.argsort(rf_imp)
axes[1].barh([feat_names[i] for i in rf_sorted],
             [rf_imp[i] for i in rf_sorted],
             color=[colors_rf[i] for i in rf_sorted],
             edgecolor='white')
axes[1].set_title('Random Forest\nFeature Importance (더 안정적)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('중요도', fontsize=11)
axes[1].set_xlim(0, max(rf_imp) * 1.3)
axes[1].grid(True, alpha=0.3, axis='x')
for i, (feat, val) in enumerate(zip([feat_names[j] for j in rf_sorted],
                                     [rf_imp[j] for j in rf_sorted])):
    axes[1].text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=11)

plt.suptitle('DT vs RF — Feature Importance 비교\n(어떤 Feature가 당뇨 위험 예측에 중요했나?)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print()
print('🏥 보건의료 관점에서 생각해보기:')
print(f'  → 1위 Feature인 "{feat_names[np.argmax(rf_imp)]}"이(가)')
print('     당뇨 위험 예측에 가장 큰 영향을 미칩니다.')
print('  → 실제 임상에서도 이 수치가 중요하게 다뤄지나요?')

---
## ✅ 실습 완료!

### 오늘 배운 것 정리

| 단계 | 핵심 내용 |
|------|-----------|
| **전처리** | 4주차와 동일 — 결측값 → 평균, 범주형 → 인코딩 |
| **DT vs RF** | 코드 딱 한 줄 차이인데 성능은 크게 다름 |
| **n_estimators** | 나무 많을수록 안정적, 과적합 없음 |
| **Feature Importance** | RF가 DT보다 더 고르게, 안정적으로 중요도 분배 |
| **보건의료 해석** | 혈당수치가 당뇨 위험 예측의 핵심 Feature |

---

### 🚀 다음 시간 예고
**실전 프로젝트** — 내가 직접 데이터를 고르고 모델을 선택합니다.  
지금까지 배운 Decision Tree와 Random Forest를 자유롭게 활용해보세요!

---
*AI 기초 강의 | 5주차 실습 완료* 🎉